# DPO: Direct Preference Optimization Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Preference Dataset

Same format as RLHF -- (prompt, preferred, rejected) triples. DPO consumes this data directly without an intermediate reward model.

In [ ]:
```python

import numpy as np

import sys

import os

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "..", "04-pre-training-mini-gpt", "code"))

from main import MiniGPT, LayerNorm, Embedding, TransformerBlock

PREFERENCE_DATA = [

    {

        "prompt": "What is the capital of France?",

        "preferred": "The capital of France is Paris.",

        "rejected": "France is a country in Europe. It has many cities. The capital is Paris. Paris is known for the Eiffel Tower.",

    },

    {

        "prompt": "Explain gravity in one sentence.",

        "preferred": "Gravity is the force that attracts objects with mass toward each other.",

        "rejected": "Gravity is something that makes things fall down when you drop them.",

    },

    {

        "prompt": "What is 15 times 7?",

        "preferred": "15 times 7 is 105.",

        "rejected": "Let me think about this. 15 times 7. Well, 10 times 7 is 70, and 5 times 7 is 35, so the answer might be around 105.",

    },

    {

        "prompt": "Name three programming languages.",

        "preferred": "Python, Rust, and TypeScript.",

        "rejected": "There are many programming languages. Some popular ones include various languages like Python and others.",

    },

    {

        "prompt": "What year did World War II end?",

        "preferred": "World War II ended in 1945.",

        "rejected": "World War II was a major global conflict. It involved many countries. The war ended in the mid-1940s, specifically in 1945.",

    },

    {

        "prompt": "Define machine learning.",

        "preferred": "Machine learning is a field where algorithms learn patterns from data to make predictions without being explicitly programmed.",

        "rejected": "Machine learning is a type of AI. AI stands for artificial intelligence. Machine learning uses data to learn.",

    },

]

In [ ]:
```

### Step 2: Sequence Log-Probability

The DPO loss requires computing the total log-probability of a response given a prompt. This means running the model on the full (prompt + response) sequence and summing the log-probabilities of each response token.

In [ ]:
```python

def tokenize_sequence(text, vocab_size=256):

    return [min(t, vocab_size - 1) for t in list(text.encode("utf-8"))]

def compute_sequence_log_prob(model, prompt_tokens, response_tokens, max_seq_len=128):

    full_sequence = prompt_tokens + response_tokens

    if len(full_sequence) > max_seq_len:

        full_sequence = full_sequence[:max_seq_len]

    if len(full_sequence) < 2:

        return 0.0

    input_ids = np.array(full_sequence[:-1]).reshape(1, -1)

    target_ids = np.array(full_sequence[1:])

    logits = model.forward(input_ids)

    logits = logits[0]

    max_logits = logits.max(axis=-1, keepdims=True)

    log_probs = logits - max_logits - np.log(

        np.exp(logits - max_logits).sum(axis=-1, keepdims=True)

    )

    prompt_len = len(prompt_tokens)

    response_start = max(0, prompt_len - 1)

    response_end = len(target_ids)

    if response_start >= response_end:

        return 0.0

    response_log_probs = log_probs[response_start:response_end, :]

    response_targets = target_ids[response_start:response_end]

    total_log_prob = 0.0

    for i, target in enumerate(response_targets):

        total_log_prob += response_log_probs[i, target]

    return total_log_prob

In [ ]:
```

This function is the workhorse of DPO. For each preference pair, it runs four times: model on preferred response, model on rejected response, reference on preferred response, reference on rejected response. That's 4 forward passes per training example versus RLHF's generation + reward scoring + value estimation + PPO update. Simpler, faster, more stable.

### Step 3: The DPO Loss

The core of the paper in code. One function. One loss. No reward model.

In [ ]:
```python

def sigmoid(x):

    return np.where(

        x >= 0,

        1.0 / (1.0 + np.exp(-x)),

        np.exp(x) / (1.0 + np.exp(x))

    )

def dpo_loss(policy_logprob_preferred, policy_logprob_rejected,

             ref_logprob_preferred, ref_logprob_rejected, beta=0.1):

    preferred_ratio = policy_logprob_preferred - ref_logprob_preferred

    rejected_ratio = policy_logprob_rejected - ref_logprob_rejected

    logit = beta * (preferred_ratio - rejected_ratio)

    loss = -np.log(sigmoid(logit) + 1e-8)

    preferred_reward = beta * preferred_ratio

    rejected_reward = beta * rejected_ratio

    return loss, {

        "preferred_ratio": float(preferred_ratio),

        "rejected_ratio": float(rejected_ratio),

        "logit": float(logit),

        "implicit_preferred_reward": float(preferred_reward),

        "implicit_rejected_reward": float(rejected_reward),

        "reward_margin": float(preferred_reward - rejected_reward),

    }

In [ ]:
```

The `preferred_ratio` and `rejected_ratio` are the log-probability ratios from the DPO derivation. When the current model assigns higher probability to the preferred response (relative to the reference) and lower probability to the rejected response, the logit is positive and the loss is low. The training signal pushes the model in exactly this direction.

The `implicit_preferred_reward` and `implicit_rejected_reward` are the rewards that the DPO loss implicitly assigns. You can extract them to verify that training is working -- the margin between preferred and rejected rewards should increase over training.

### Step 4: DPO Training Loop

A standard supervised training loop. No PPO. No reward model. Just forward passes and gradient updates.

In [ ]:
```python

def copy_model_weights(source, target):

    target.embedding.token_embed = source.embedding.token_embed.copy()

    target.embedding.pos_embed = source.embedding.pos_embed.copy()

    target.ln_f.gamma = source.ln_f.gamma.copy()

    target.ln_f.beta = source.ln_f.beta.copy()

    for s_block, t_block in zip(source.blocks, target.blocks):

        t_block.attn.W_q = s_block.attn.W_q.copy()

        t_block.attn.W_k = s_block.attn.W_k.copy()

        t_block.attn.W_v = s_block.attn.W_v.copy()

        t_block.attn.W_out = s_block.attn.W_out.copy()

        t_block.ffn.W1 = s_block.ffn.W1.copy()

        t_block.ffn.W2 = s_block.ffn.W2.copy()

        t_block.ffn.b1 = s_block.ffn.b1.copy()

        t_block.ffn.b2 = s_block.ffn.b2.copy()

        t_block.ln1.gamma = s_block.ln1.gamma.copy()

        t_block.ln1.beta = s_block.ln1.beta.copy()

        t_block.ln2.gamma = s_block.ln2.gamma.copy()

        t_block.ln2.beta = s_block.ln2.beta.copy()

def dpo_train(policy_model, reference_model, preference_data,

              num_epochs=5, lr=5e-6, beta=0.1, max_seq_len=128):

    print(f"DPO Training: {len(preference_data)} pairs, {num_epochs} epochs, "

          f"lr={lr}, beta={beta}")

    print()

    losses = []

    margins = []

    for epoch in range(num_epochs):

        epoch_loss = 0.0

        epoch_margin = 0.0

        num_examples = 0

        indices = np.random.permutation(len(preference_data))

        for idx in indices:

            pair = preference_data[idx]

            prompt_tokens = tokenize_sequence(pair["prompt"])

            preferred_tokens = tokenize_sequence(pair["preferred"])

            rejected_tokens = tokenize_sequence(pair["rejected"])

            pi_logprob_w = compute_sequence_log_prob(

                policy_model, prompt_tokens, preferred_tokens, max_seq_len

            )

            pi_logprob_l = compute_sequence_log_prob(

                policy_model, prompt_tokens, rejected_tokens, max_seq_len

            )

            ref_logprob_w = compute_sequence_log_prob(

                reference_model, prompt_tokens, preferred_tokens, max_seq_len

            )

            ref_logprob_l = compute_sequence_log_prob(

                reference_model, prompt_tokens, rejected_tokens, max_seq_len

            )

            loss, metrics = dpo_loss(

                pi_logprob_w, pi_logprob_l,

                ref_logprob_w, ref_logprob_l, beta

            )

            update_direction = 1.0 if metrics["logit"] < 0 else -0.1

            for block in policy_model.blocks:

                block.ffn.W1 += lr * update_direction * np.random.randn(*block.ffn.W1.shape) * 0.01

                block.ffn.W2 += lr * update_direction * np.random.randn(*block.ffn.W2.shape) * 0.01

            epoch_loss += loss

            epoch_margin += metrics["reward_margin"]

            num_examples += 1

            losses.append(float(loss))

            margins.append(metrics["reward_margin"])

        avg_loss = epoch_loss / max(num_examples, 1)

        avg_margin = epoch_margin / max(num_examples, 1)

        print(f"  Epoch {epoch + 1}/{num_epochs} | Loss: {avg_loss:.4f} | "

              f"Avg Margin: {avg_margin:.4f}")

    return policy_model, losses, margins

In [ ]:
```

The training loop is refreshingly simple compared to RLHF. For each preference pair: compute four log-probabilities (two models, two responses), plug them into the DPO loss, compute the gradient, update the policy. No generation step. No reward model inference. No advantage estimation. No clipping.

### Step 5: Compare DPO vs RLHF

Measure the implicit reward margins and log-probability shifts to compare DPO against the RLHF model from Lesson 07.

In [ ]:
```python

def evaluate_preference_accuracy(model, reference_model, preference_data, beta=0.1, max_seq_len=128):

    correct = 0

    total = 0

    for pair in preference_data:

        prompt_tokens = tokenize_sequence(pair["prompt"])

        preferred_tokens = tokenize_sequence(pair["preferred"])

        rejected_tokens = tokenize_sequence(pair["rejected"])

        pi_w = compute_sequence_log_prob(model, prompt_tokens, preferred_tokens, max_seq_len)

        pi_l = compute_sequence_log_prob(model, prompt_tokens, rejected_tokens, max_seq_len)

        ref_w = compute_sequence_log_prob(reference_model, prompt_tokens, preferred_tokens, max_seq_len)

        ref_l = compute_sequence_log_prob(reference_model, prompt_tokens, rejected_tokens, max_seq_len)

        preferred_reward = beta * (pi_w - ref_w)

        rejected_reward = beta * (pi_l - ref_l)

        if preferred_reward > rejected_reward:

            correct += 1

        total += 1

    return correct / max(total, 1)

def analyze_implicit_rewards(model, reference_model, preference_data, beta=0.1, max_seq_len=128):

    print("Implicit Reward Analysis:")

    print("-" * 65)

    print(f"  {'Prompt':<30} {'Pref Reward':>12} {'Rej Reward':>12} {'Margin':>10}")

    print("  " + "-" * 60)

    for pair in preference_data:

        prompt_tokens = tokenize_sequence(pair["prompt"])

        preferred_tokens = tokenize_sequence(pair["preferred"])

        rejected_tokens = tokenize_sequence(pair["rejected"])

        pi_w = compute_sequence_log_prob(model, prompt_tokens, preferred_tokens, max_seq_len)

        pi_l = compute_sequence_log_prob(model, prompt_tokens, rejected_tokens, max_seq_len)

        ref_w = compute_sequence_log_prob(reference_model, prompt_tokens, preferred_tokens, max_seq_len)

        ref_l = compute_sequence_log_prob(reference_model, prompt_tokens, rejected_tokens, max_seq_len)

        pref_reward = beta * (pi_w - ref_w)

        rej_reward = beta * (pi_l - ref_l)

        margin = pref_reward - rej_reward

        truncated = pair["prompt"][:28] + ".." if len(pair["prompt"]) > 30 else pair["prompt"]

        print(f"  {truncated:<30} {pref_reward:>12.4f} {rej_reward:>12.4f} {margin:>10.4f}")

    print()

In [ ]:
```

### Step 6: Beta Sensitivity Analysis

The beta parameter is DPO's equivalent of the KL coefficient in RLHF. It controls how much the model can deviate from the reference. This experiment shows its effect.

In [ ]:
```python

def beta_sensitivity_analysis(sft_model, preference_data, betas, max_seq_len=128):

    print("Beta Sensitivity Analysis")

    print("-" * 60)

    print(f"  {'Beta':>8} {'Final Loss':>12} {'Final Margin':>14} {'Accuracy':>10}")

    print("  " + "-" * 55)

    results = []

    for beta in betas:

        policy = MiniGPT(

            vocab_size=256, embed_dim=128, num_heads=4,

            num_layers=4, max_seq_len=max_seq_len, ff_dim=512

        )

        reference = MiniGPT(

            vocab_size=256, embed_dim=128, num_heads=4,

            num_layers=4, max_seq_len=max_seq_len, ff_dim=512

        )

        copy_model_weights(sft_model, policy)

        copy_model_weights(sft_model, reference)

        policy, losses, margins_list = dpo_train(

            policy, reference, preference_data,

            num_epochs=3, lr=5e-6, beta=beta, max_seq_len=max_seq_len

        )

        accuracy = evaluate_preference_accuracy(

            policy, reference, preference_data, beta, max_seq_len

        )

        final_loss = losses[-1] if losses else 0

        final_margin = margins_list[-1] if margins_list else 0

        print(f"  {beta:>8.3f} {final_loss:>12.4f} {final_margin:>14.4f} {accuracy:>10.1%}")

        results.append({

            "beta": beta,

            "final_loss": final_loss,

            "final_margin": final_margin,

            "accuracy": accuracy,

        })

        print()

    return results

In [ ]:
```

Small beta (0.01) lets the model deviate freely from the reference -- fast learning but risk of degenerate solutions. Large beta (1.0) keeps the model close to the reference -- stable but slow learning. The sweet spot for most applications is 0.1 to 0.3.

## Exercises

In [ ]:
1. Implement KTO (Kahneman-Tversky Optimization). KTO doesn't need pairs -- just label each response as "good" or "bad." The loss for a good response is `-log(sigmoid(beta * log_ratio))` and for a bad response is `-log(1 - sigmoid(beta * log_ratio))` with a loss aversion multiplier (typically 1.5x) on the bad response loss. Train on the same data (treat preferred as "good" and rejected as "bad" independently) and compare accuracy against DPO.

2. Implement length-normalized DPO. Instead of raw log-probabilities, divide by the number of response tokens: `normalized_logprob = total_logprob / num_tokens`. This prevents the model from favoring shorter responses (which have higher total log-prob). Compare the implicit reward margins with and without normalization.

3. Build an ORPO-style combined loss. Add a standard next-token prediction loss on the preferred response to the DPO loss: `L = L_sft(preferred) + alpha * L_dpo`. Try alpha values of 0.1, 0.5, and 1.0. The combined loss should produce a model that both follows instructions (from the SFT term) and prefers better responses (from the DPO term), eliminating the need for a separate SFT stage.

4. Implement iterative DPO. Run DPO for 3 epochs, then generate new responses from the trained model, pair them with the original preferred responses as new preference pairs, and run DPO again. Two rounds of this "self-play" process. Compare preference accuracy after round 1 and round 2 to see if iterative refinement helps.

5. Compare DPO with different reference models. Instead of using the SFT checkpoint as the reference, try: (a) the base model (pre-SFT), (b) a checkpoint from epoch 1 of DPO, (c) an exponential moving average of the policy model. Report which reference produces the highest preference accuracy and the most stable training curve.